# Semantic LLM Cache with LangChain-enVector

This example shows `EnvectorSemanticCache`, a LangChain LLM cache that keeps the
prompts it has seen and their answers in an enVector index. A new prompt that is
close enough in meaning to a stored one is answered from the cache, and nothing
stored is readable by the server.

In this example, we'll see:

- How a paraphrased question is served from the cache without a model request
- How the prompt embeddings, the prompt text and the answers stay encrypted in the index
- How answers from different model settings are kept apart
- How to clear the cache

## Prerequisites
- enVector server reachable from this notebook environment
- A key path for the key this notebook registers (it is created if missing)
- Ollama running locally, with internet access to pull the two models used below
- `langchain-envector` and `langchain-ollama` packages installed

### Import langchain-envector

Import `langchain_envector` to use enVector with the LangChain framework.

In [ ]:
# !pip install langchain-envector langchain-ollama

In [ ]:
import langchain_envector

## Configure connection and models
Fill in your enVector connection details. The embedding model turns prompts into
vectors; the chat model answers them. Both run locally through Ollama here, so
no API key is needed.

In [ ]:
import os

address = os.getenv("ENVECTOR_ADDRESS", "0.0.0.0:50050")
token = os.getenv("ENVECTOR_ACCESS_TOKEN", "")

key_path = "./keys"
key_id = "semantic_cache_key"
index_name = "llm_cache_demo"
embedding_model = "all-minilm:l6-v2"
chat_model = "gemma3:270m"

print(f"Using key '{key_id}'")
print(f"Using index '{index_name}'")
print(f"Embedding model: {embedding_model}")
print(f"Chat model: {chat_model}")

## Pull the Ollama models
Pulling them up front keeps the timings below about the cache rather than about downloads.

In [ ]:
!ollama pull {embedding_model}
!ollama pull {chat_model}

## Set up the embedding model and the chat model
The embedding model's output size becomes the index dimension. The chat model
answers the questions; `temperature=0` keeps its answers stable.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

embeddings = OllamaEmbeddings(model=embedding_model)
vector_dim = len(embeddings.embed_query("dimension probe"))
print(f"Embedding dimension: {vector_dim}")

llm = ChatOllama(model=chat_model, temperature=0)

## Create the cache and register it with LangChain

`EnvectorSemanticCache` takes the same configuration as the `Envector` vector
store. Two settings matter here:

- `IndexSettings.metadata_encryption=True` encrypts the prompt text and the stored
  answers. The prompt embeddings are encrypted in any case.
- `similarity_threshold` is the cosine similarity a stored prompt needs to count
  as a hit: `1.0` means an identical prompt, and higher is stricter.

`set_llm_cache` makes every LangChain model request go through this cache from
now on.

In [ ]:
from langchain_core.globals import set_llm_cache

from langchain_envector import EnvectorSemanticCache
from langchain_envector.config import ConnectionConfig, EnvectorConfig, IndexSettings, KeyConfig

config = EnvectorConfig(
    connection=ConnectionConfig(address=address, access_token=token) if token else ConnectionConfig(address=address),
    key=KeyConfig(key_path=key_path, key_id=key_id),
    index=IndexSettings(index_name=index_name, dim=vector_dim, metadata_encryption=True),
    create_if_missing=True,
)
cache = EnvectorSemanticCache(config=config, embeddings=embeddings, similarity_threshold=0.9)
set_llm_cache(cache)

## Ask a question, then ask it differently

The first question goes to the model and its answer is stored. The second
question is worded differently but means the same, so the cache serves the
stored answer without a model request. The third question is about something
else and goes to the model again.

In [ ]:
import time


def ask(question: str, **request_settings):
    started = time.time()
    answer = llm.invoke(question, **request_settings).content
    print(f"[{time.time() - started:5.2f} s] {question}")
    print(f"          {answer.strip()[:120]}")


ask("What is the capital of France?")
ask("Tell me the capital city of France.")
ask("What is the capital of Germany?")

The second call is answered from the cache: it returns in a fraction of the
time and with the same text as the first, while the first and the third took
a model request each. Had the reworded question scored below
`similarity_threshold`, it would have gone to the model like the third one.

## Different request settings do not share answers

Along with the prompt, LangChain hands the cache a string describing the model
and the settings of the request (`llm_string`). The cache keeps one enVector
partition per such string, so a request with other settings cannot receive an
answer produced under different ones, even for the very same question.

Which settings go into `llm_string` is decided by the chat model class.
`ChatOllama` puts in only what is passed with the request, not its model name or
constructor arguments, so here the temperature is passed per request through
Ollama's `options`.

In [ ]:
ask("What is the capital of France?", options={"temperature": 0.8})

for partition in cache.vectorstore.list_partitions():
    if partition["name"].startswith("lc_cache_"):
        print(partition)

The call took a model request although the same question was cached for the
default settings, and two partitions now exist, one per request configuration.

## Clear the cache

`cache.clear()` drops every partition this cache created and leaves the rest of
the index alone. After it, the paraphrase from above is a miss again.

In [ ]:
cache.clear()

ask("Tell me the capital city of France.")

### Clean Up

We can delete the created index and the registered key when they are no longer needed.

In [ ]:
cache.vectorstore.client.ev.drop_index(index_name)

In [ ]:
cache.vectorstore.client.ev.delete_key(key_id)